In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch torchvision torchaudio opencv-python pandas

In [ ]:
import os
drive_path = "/content/drive/MyDrive/Proiect_Dino"

if not os.path.exists(drive_path):
    os.makedirs(drive_path)

%cd {drive_path}

/content/drive/MyDrive/Proiect_Dino


In [ ]:
repo_name = "dino_wm"
if not os.path.exists(repo_name):
    !git clone https://github.com/gaoyuezhou/dino_wm
%cd {repo_name}

Cloning into 'dino_wm'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 257 (delta 35), reused 24 (delta 24), pack-reused 160 (from 1)
Receiving objects: 100% (257/257), 5.56 MiB | 7.60 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (190/190), done.
/content/drive/MyDrive/Proiect_Dino/dino_wm


In [ ]:
%cd /content/drive/MyDrive/Proiect_Dino/dino_wm

/content/drive/MyDrive/Proiect_Dino/dino_wm


In [ ]:
import cv2
import os

BASE_DIR = '/content/drive/MyDrive/Proiect_Dino/dino_wm'

video_path = os.path.join(BASE_DIR, 'datasets/raw_video/User_15_Short_10.mp4')
output_folder = os.path.join(BASE_DIR, 'datasets/raw_video/User15')
target_fps = 10

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"CRITICAL: Nu pot deschide fișierul la calea: {video_path}")
    print("Verifică dacă fișierul există în Drive la această adresă!")
else:
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    hop = max(1, round(video_fps / target_fps))
    print(f"Succes: Video deschis. FPS: {video_fps} | Hop: {hop}")

    count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % hop == 0:
            file_name = f"frame_{saved_count:04d}.jpg"
            save_path = os.path.join(output_folder, file_name)
            cv2.imwrite(save_path, frame)
            saved_count += 1

        count += 1

    cap.release()
    print(f"Finalizat! S-au salvat {saved_count} cadre în {output_folder}")

Succes: Video deschis. FPS: 10.0 | Hop: 1
Finalizat! S-au salvat 330 cadre în /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/raw_video/User15


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
import cv2
from pathlib import Path

print("All imports OK")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")


All imports OK
PyTorch version : 2.10.0+cu128
CUDA available  : True


In [ ]:
import os
import matplotlib.cm as cm

BASE_DIR = '/content/drive/MyDrive/Proiect_Dino/dino_wm'

# Input paths
VIDEO_PATH    = os.path.join(BASE_DIR, 'datasets/raw_video/User15.mp4')
FRAMES_FOLDER = os.path.join(BASE_DIR, 'datasets/raw_video/User15')

# Output paths
OUT_ROOT       = os.path.join(BASE_DIR, 'datasets/attention_output')
OUT_SINGLE     = os.path.join(OUT_ROOT, 'single_frames')
OUT_BATCH      = os.path.join(OUT_ROOT, 'batch')
OUT_COMPARISON = os.path.join(OUT_ROOT, 'comparison')
OUT_CSV        = os.path.join(OUT_ROOT, 'attention_results.csv')

# Video / model settings
PATCH_SIZE  = 14
IMG_SIZE    = 224
N_PATCHES   = IMG_SIZE // PATCH_SIZE

# Heatmap display settings
BLEND_ALPHA = 0.55
COLORMAP    = cm.inferno

# Batch settings
SEGMENT_SIZE = 15

for folder in [OUT_SINGLE, OUT_BATCH, OUT_COMPARISON]:
    os.makedirs(folder, exist_ok=True)

print("Config loaded.")
print(f"  Frames folder : {FRAMES_FOLDER}")
print(f"  Output root   : {OUT_ROOT}")
print(f"  Patch size    : {PATCH_SIZE}")
print(f"  Patch grid    : {N_PATCHES}×{N_PATCHES} = {N_PATCHES**2} patches")

Config loaded.
  Frames folder : /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/raw_video/User15
  Output root   : /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output
  Patch size    : 14
  Patch grid    : 16×16 = 256 patches


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model = model.to(device).eval()

# Image transform
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

print("Model ready.")

Using device: cuda
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 238MB/s]


Model ready.


In [ ]:
def get_cls_attention(img_pil):
    attention_store = []

    def hook_fn(module, input, output):
        attention_store.append(output.detach())

    # We compute attention weights manually from the raw qkv output.
    hook = model.blocks[-1].attn.qkv.register_forward_hook(hook_fn)

    tensor = transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        _ = model(tensor)
    hook.remove()

    qkv = attention_store[0]  # (1, seq_len, 3 * heads * head_dim)
    B, N, _ = qkv.shape       # N = 257 (1 CLS + 256 patches)

    num_heads = model.blocks[-1].attn.num_heads
    head_dim  = qkv.shape[-1] // (3 * num_heads)

    # Split into Q, K, V
    qkv   = qkv.reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
    q, k  = qkv[0], qkv[1]   # each: (1, heads, 257, head_dim)

    # Compute scaled dot-product attention weights
    scale = head_dim ** -0.5
    attn  = (q @ k.transpose(-2, -1)) * scale   # (1, heads, 257, 257)
    attn  = attn.softmax(dim=-1)

    # CLS token (index 0) → all patches (index 1:)
    cls_to_patches = attn[0, :, 0, 1:]          # (heads, 256)

    attn_map   = cls_to_patches.mean(0).reshape(N_PATCHES, N_PATCHES).cpu().numpy()
    attn_heads = cls_to_patches.reshape(-1, N_PATCHES, N_PATCHES).cpu().numpy()

    return attn_map, attn_heads

def make_overlay(img_pil, attn_16x16, alpha=BLEND_ALPHA, cmap=COLORMAP):
    """
    Upsample 16×16 attention to 224×224, apply colormap, blend with image.
    Returns uint8 numpy array (224, 224, 3).
    """
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0

    # Bicubic upsampling gives smooth transitions (better than nearest-neighbour)
    attn_up   = cv2.resize(attn_16x16, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    attn_norm = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)

    heatmap = cmap(attn_norm)[:, :, :3]  # RGBA → RGB
    blended = (1 - alpha) * img_np + alpha * heatmap
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)


def make_head_grid(attn_heads):
    """
    Stack all per-head attention maps into one image for display.
    ViT-S has 6 heads → 2×3 or 3×2 grid.
    """
    n_heads   = attn_heads.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_heads)))
    grid_rows = int(np.ceil(n_heads / grid_cols))
    grid      = np.zeros((grid_rows * N_PATCHES, grid_cols * N_PATCHES))

    for h, head_map in enumerate(attn_heads):
        r, c  = divmod(h, grid_cols)
        norm  = (head_map - head_map.min()) / (head_map.max() - head_map.min() + 1e-8)
        grid[r*N_PATCHES:(r+1)*N_PATCHES, c*N_PATCHES:(c+1)*N_PATCHES] = norm

    return grid


def attn_entropy(attn_map):
    """
    Shannon entropy. Measures how SPREAD the attention is.
    High entropy → attention uniform over many patches (unfocused scene).
    Low entropy  → attention concentrated on a few patches (clear object).
    Max possible for 256 patches = log(256) ≈ 5.55
    """
    p = attn_map.flatten()
    p = p / (p.sum() + 1e-8)
    p = p[p > 0]
    return float(-np.sum(p * np.log(p + 1e-8)))

def top_k_mass(attn_map, k=10):
    """
    Fraction of total attention in the top-k patches.
    e.g. 0.45 means 45% of all attention sits in just 10 out of 256 patches.
    Higher = more concentrated.
    """
    flat = attn_map.flatten()
    return float(np.sort(flat)[::-1][:k].sum() / (flat.sum() + 1e-8))

def peak_patch(attn_map):
    """Row, col of the single patch with maximum attention."""
    idx = attn_map.argmax()
    return divmod(int(idx), N_PATCHES)  # (row, col)

def compute_metrics(attn_map, label=''):
    return {
        'label':        label,
        'entropy':      round(attn_entropy(attn_map), 4),
        'top10_mass':   round(top_k_mass(attn_map, k=10), 4),
        'top10_mass_%': round(top_k_mass(attn_map, k=10) * 100, 1),
        'peak_row':     peak_patch(attn_map)[0],
        'peak_col':     peak_patch(attn_map)[1],
        'attn_max':     round(float(attn_map.max()), 6),
        'attn_mean':    round(float(attn_map.mean()), 6),
    }


def load_frame_paths():
    paths = sorted([
        os.path.join(FRAMES_FOLDER, f)
        for f in os.listdir(FRAMES_FOLDER)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    print(f"Found {len(paths)} frames in {FRAMES_FOLDER}")
    return paths

print("Helper functions defined.")



Helper functions defined.


In [ ]:
def run_single_frame_analysis():

    global VIDEO_FPS, OUT_SINGLE, IMG_SIZE


    if 'VIDEO_FPS' not in globals():
        print("Atenție: VIDEO_FPS nu a fost găsit, setez valoarea implicită 5.")
        VIDEO_FPS = 5

    Path(OUT_SINGLE).mkdir(parents=True, exist_ok=True)
    frame_paths = load_frame_paths()
    records     = []

    for idx, fpath in enumerate(frame_paths):
        img = Image.open(fpath).convert('RGB')

        # Extract attention
        attn_map, attn_heads = get_cls_attention(img)

        # Build 3-panel figure
        fig = plt.figure(figsize=(15, 5), facecolor='#0d0d0d')
        gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

        # Panel 1
        ax1 = fig.add_subplot(gs[0])
        ax1.imshow(img.resize((IMG_SIZE, IMG_SIZE)))
        ax1.set_title('Original frame', color='white', fontsize=11, pad=6)
        ax1.axis('off')

        # Panel 2
        ax2 = fig.add_subplot(gs[1])
        overlay = make_overlay(img, attn_map)
        ax2.imshow(overlay)
        m = compute_metrics(attn_map)
        ax2.set_title(
            f'CLS attention  |  entropy={m["entropy"]:.2f}  |  top-10={m["top10_mass_%"]:.0f}%',
            color='white', fontsize=10, pad=6
        )
        ax2.axis('off')

        # Panel 3 — per-head grid
        ax3 = fig.add_subplot(gs[2])
        head_grid = make_head_grid(attn_heads)
        ax3.imshow(head_grid, cmap='inferno', interpolation='nearest')
        ax3.set_title(f'Per-head ({attn_heads.shape[0]} heads)', color='white', fontsize=11, pad=6)
        ax3.axis('off')

        fname = os.path.basename(fpath)
        fig.suptitle(f'Frame {idx:04d} — {fname}  |  t={idx/VIDEO_FPS:.1f}s',
                     color='#aaaaaa', fontsize=10, y=1.01)
        plt.tight_layout(pad=0.3)

        out_path = os.path.join(OUT_SINGLE, f'frame_{idx:04d}.png')
        fig.savefig(out_path, dpi=110, bbox_inches='tight', facecolor='#0d0d0d')
        plt.close(fig)

        # Save raw attention map
        np.save(os.path.join(OUT_SINGLE, f'frame_{idx:04d}_attn.npy'), attn_map)

        # Record metrics
        row = compute_metrics(attn_map, label=f'frame_{idx:04d}')
        row['frame_id']      = idx
        row['frame_file']    = fname
        row['timestamp_sec'] = round(idx / VIDEO_FPS, 3)
        records.append(row)

        print(f"  [{idx+1:3d}/{len(frame_paths)}] entropy={m['entropy']:.2f}  "
              f"top10={m['top10_mass_%']:.0f}%  "
              f"peak=({m['peak_row']},{m['peak_col']})", end='\r')

    # Save per-frame CSV
    df = pd.DataFrame(records)
    csv_path = os.path.join(OUT_SINGLE, 'single_frame_metrics.csv')
    df.to_csv(csv_path, index=False)

    print(f"\n\n{'='*55}")
    print("SINGLE-FRAME RESULTS")
    print('='*55)
    print(f"  Saved heatmaps → {OUT_SINGLE}")
    print(f"  Saved metrics  → {csv_path}")
    print('='*55)

    return df

df_single = run_single_frame_analysis()

Atenție: VIDEO_FPS nu a fost găsit, setez valoarea implicită 5.
Found 330 frames in /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/raw_video/User15


/tmp/ipykernel_896/4139641104.py:52: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0.3)


  [330/330] entropy=5.07  top10=21%  peak=(9,3)

SINGLE-FRAME RESULTS
  Saved heatmaps → /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/single_frames
  Saved metrics  → /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/single_frames/single_frame_metrics.csv


In [ ]:
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Proiect_Dino/dino_wm/datasets'

FRAMES_FOLDER   = f'{BASE_PATH}/raw_video/User15'
HAND_CSV        = f'{BASE_PATH}/wrist_and_palm_poses.csv'
OUT_ROOT        = f'{BASE_PATH}/attention_output'
OUT_GUIDED      = f'{OUT_ROOT}/single_frames_guided'
OUT_COMP        = f'{OUT_ROOT}/comparison'

Path(OUT_GUIDED).mkdir(parents=True, exist_ok=True)
Path(OUT_COMP).mkdir(parents=True, exist_ok=True)

VIDEO_FPS, IMG_SIZE, N_PATCHES = 5, 224, 16
SIGMA_PATCHES, GUIDE_STRENGTH, CONF_THRESHOLD = 2.0, 6.0, 0.5
BLEND_ALPHA, COLORMAP = 0.55, plt.cm.inferno


hand_df = pd.read_csv(HAND_CSV)
t0 = hand_df['tracking_timestamp_us'].iloc[0]
hand_df['time_sec'] = (hand_df['tracking_timestamp_us'] - t0) / 1e6

def nearest_hand_row(t_sec):
    idx = (hand_df['time_sec'] - t_sec).abs().idxmin()
    return hand_df.iloc[idx]

def device_to_patch(x_dev, y_dev):
    px = (x_dev / 1.5 + 0.5) * N_PATCHES
    py = (-y_dev / 1.5 + 0.5) * N_PATCHES
    return float(np.clip(px, 0, N_PATCHES - 0.01)), float(np.clip(py, 0, N_PATCHES - 0.01))

def gaussian_weight_map(points):
    gmap = np.zeros((N_PATCHES, N_PATCHES), dtype=np.float32)
    if not points: return gmap
    rows, cols = np.mgrid[0:N_PATCHES, 0:N_PATCHES].astype(np.float32)
    for (cx, cy) in points:
        gmap += np.exp(-((cols - cx)**2 + (rows - cy)**2) / (2 * SIGMA_PATCHES**2))
    return gmap / (gmap.max() + 1e-8)

def apply_hand_guidance(raw_attn, hand_row):
    points = []

    for conf, x, y in [('left_tracking_confidence', 'tx_left_wrist_device', 'ty_left_wrist_device'),
                       ('right_tracking_confidence', 'tx_right_wrist_device', 'ty_right_wrist_device')]:
        if float(hand_row.get(conf, 0)) >= CONF_THRESHOLD:
            points.append(device_to_patch(float(hand_row[x]), float(hand_row[y])))

    hand_map = gaussian_weight_map(points)
    guided = raw_attn * (1.0 + GUIDE_STRENGTH * hand_map) if points else raw_attn
    return guided / (guided.sum() + 1e-8)

def make_overlay_local(img_pil, attn_16x16):
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32)/255.
    attn_up = cv2.resize(attn_16x16, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    attn_n = (attn_up - attn_up.min()) / (attn_up.max()-attn_up.min()+1e-8)
    heatmap = COLORMAP(attn_n)[:,:,:3]
    return (np.clip((1-BLEND_ALPHA)*img_np + BLEND_ALPHA*heatmap, 0, 1)*255).astype(np.uint8)

frame_files = sorted([f for f in os.listdir(FRAMES_FOLDER) if f.lower().endswith(('.jpg','.png'))])

print(f"Încep procesarea a {len(frame_files)} frame-uri...")
for idx, fname in enumerate(frame_files):
    img = Image.open(os.path.join(FRAMES_FOLDER, fname)).convert('RGB')
    raw_attn, _ = get_cls_attention(img)

    guided_attn = apply_hand_guidance(raw_attn, nearest_hand_row(idx / VIDEO_FPS))

    ov = make_overlay_local(img, guided_attn)
    Image.fromarray(ov).save(os.path.join(OUT_GUIDED, f'frame_{idx:04d}_guided.png'))
    print(f"Procesat: {idx+1}/{len(frame_files)}", end='\r')

print(f"\nFinalizat! Rezultatele sunt salvate în: {OUT_ROOT}")

Încep procesarea a 330 frame-uri...
Procesat: 330/330
Finalizat! Rezultatele sunt salvate în: /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output


In [ ]:


import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from collections import deque
import torch
from transformers import AutoImageProcessor, AutoModel, AutoConfig
import zipfile
from google.colab import files
from projectaria_tools.core import data_provider

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR      = '/content/drive/MyDrive/Proiect_Dino/dino_wm'
FRAMES_FOLDER = f'{BASE_DIR}/datasets/raw_video/User15'
HAND_CSV      = f'{BASE_DIR}/datasets/wrist_and_palm_poses.csv'
VRS_PATH      = f'{BASE_DIR}/datasets/User_15_Short_10.vrs'
JSONL_PATH    = f'{BASE_DIR}/datasets/online_calibration.jsonl'

OUT_ROOT      = f'{BASE_DIR}/datasets/attention_output'
OUT_GUIDED    = f'{OUT_ROOT}/pictures_corrected_final'
OUT_COMP      = f'{OUT_ROOT}/comparison'
Path(OUT_GUIDED).mkdir(parents=True, exist_ok=True)
Path(OUT_COMP).mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
VIDEO_FPS     = 5
IMG_SIZE      = 224
N_PATCHES     = 16
BLEND_ALPHA   = 0.55
COLORMAP      = plt.cm.inferno

FRAME_STEP    = 10

# Guidance tuning
SIGMA_PATCHES  = 2.0
GUIDE_STRENGTH = 15.0
CONF_THRESHOLD = 0.5

# Smoothing buffers (same as gaze code)
smooth_buf = {
    'left_wrist': deque(maxlen=3),
    'left_palm': deque(maxlen=3),
    'right_wrist': deque(maxlen=3),
    'right_palm': deque(maxlen=3),
}


def load_dinov2_model():
    model_name = "facebook/dinov2-base"
    processor = AutoImageProcessor.from_pretrained(model_name)
    config = AutoConfig.from_pretrained(model_name)
    config.output_attentions = True
    config.output_hidden_states = True
    model = AutoModel.from_pretrained(model_name, config=config)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
        print("Model loaded on GPU")
    else:
        print("Model loaded on CPU")
    return model, processor

dinov2_model, dinov2_processor = load_dinov2_model()

def get_cls_attention(img_pil):
    inputs = dinov2_processor(images=img_pil, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = dinov2_model(**inputs)
    if not hasattr(outputs, 'attentions') or outputs.attentions is None:
        attn_map = np.ones((N_PATCHES, N_PATCHES), dtype=np.float32)
        return attn_map / (attn_map.sum() + 1e-8)
    attentions = outputs.attentions[-1]
    if attentions.shape[-1] >= 257:
        cls_attn = attentions[0, :, 0, 1:257]
    else:
        cls_attn = attentions[0, :, 0, 1:]
    cls_attn = cls_attn.mean(dim=0)
    try:
        attn_map = cls_attn.reshape(N_PATCHES, N_PATCHES).cpu().numpy()
    except:
        side = int(np.sqrt(len(cls_attn)))
        attn_map = cls_attn.reshape(side, side).cpu().numpy()
        attn_map = cv2.resize(attn_map, (N_PATCHES, N_PATCHES), interpolation=cv2.INTER_CUBIC)
    return attn_map / (attn_map.sum() + 1e-8)


print("Loading calibration from VRS")

provider = data_provider.create_vrs_data_provider(VRS_PATH)
device_calib = provider.get_device_calibration()
rgb_calib = device_calib.get_camera_calib("camera-rgb")

# Get camera intrinsics (exact same as gaze)
FX, FY = rgb_calib.get_focal_lengths()
CX, CY = rgb_calib.get_principal_point()
native_size = rgb_calib.get_image_size()
NATIVE_W, NATIVE_H = native_size[0], native_size[1]

# Get device to RGB transform (for parallax offset)
T_device_rgb = device_calib.get_transform_device_sensor("camera-rgb")
pos_flat = T_device_rgb.translation().flatten()
OFFSET_X, OFFSET_Y = -pos_flat[0], -pos_flat[1]

# Scale to IMG_SIZE (224x224)
S_W = IMG_SIZE / NATIVE_W
S_H = IMG_SIZE / NATIVE_H
FXs = FX * S_W
FYs = FY * S_H
CXs = CX * S_W
CYs = CY * S_H

print(f"  Camera: fx={FX:.1f}, fy={FY:.1f}, cx={CX:.1f}, cy={CY:.1f}")
print(f"  Native size: {NATIVE_W}x{NATIVE_H}")
print(f"  Offset: X={OFFSET_X*1000:.2f}mm, Y={OFFSET_Y*1000:.2f}mm")
print(f"  Scaled: FXs={FXs:.2f}, FYs={FYs:.2f}, CXs={CXs:.2f}, CYs={CYs:.2f}")

# Get angular corrections from JSONL (for rotation between cameras)
def get_angular_corrections_from_jsonl(jsonl_path):
    with open(jsonl_path, 'r') as f:
        first_line = f.readline()
        data = json.loads(first_line)

    slam_left_yaw = None
    slam_left_pitch = None
    rgb_yaw = None
    rgb_pitch = None

    for cam in data['CameraCalibrations']:
        if cam['Label'] in ['camera-slam-left', 'camera-rgb']:
            quat = cam['T_Device_Camera']['UnitQuaternion']
            qw, qx, qy, qz = quat[0], quat[1][0], quat[1][1], quat[1][2]

            siny_cosp = 2.0 * (qw * qz + qx * qy)
            cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
            yaw_val = np.arctan2(siny_cosp, cosy_cosp)

            sinp = 2.0 * (qw * qy - qz * qx)
            pitch_val = np.arcsin(np.clip(sinp, -1, 1))

            if cam['Label'] == 'camera-slam-left':
                slam_left_yaw, slam_left_pitch = yaw_val, pitch_val
            else:
                rgb_yaw, rgb_pitch = yaw_val, pitch_val

    return rgb_yaw - slam_left_yaw, rgb_pitch - slam_left_pitch

YAW_CORR, PITCH_CORR = get_angular_corrections_from_jsonl(JSONL_PATH)
print(f"  Angular corrections: Yaw={np.rad2deg(YAW_CORR):.2f}°, Pitch={np.rad2deg(PITCH_CORR):.2f}°")


def project_device_point_to_pixel(tx, ty, tz):
    """
    Proiecție folosind EXACT aceeași formulă ca la eye-gaze.

    În codul de gaze:
        yaw = from CSV
        pitch = from CSV
        x_corr = (OFFSET_X / depth) * FXs
        y_corr = (OFFSET_Y / depth) * FYs
        raw_x = CXs + (FXs * np.tan(yaw)) + x_corr
        raw_y = CYs - (FYs * (np.tan(pitch) / np.cos(yaw))) - y_corr

    Pentru mâini, calculăm yaw și pitch din coordonatele XYZ:
        yaw = atan2(tx, tz)
        pitch = atan2(-ty, tz)   # minus pentru că în imagine y crește în jos
    """
    if tz <= 0.01:
        return None

    yaw = np.arctan2(tx, tz)
    pitch = np.arctan2(-ty, tz)

    yaw_corrected = yaw + YAW_CORR
    pitch_corrected = pitch + PITCH_CORR

    depth = tz

    x_parallax = (OFFSET_X / depth) * FXs
    y_parallax = (OFFSET_Y / depth) * FYs

    raw_x = CXs + (FXs * np.tan(yaw_corrected)) + x_parallax
    raw_y = CYs - (FYs * (np.tan(pitch_corrected) / max(np.cos(yaw_corrected), 1e-6))) - y_parallax

    return float(raw_x), float(raw_y)

def pixel_to_patch(px, py):
    col = (px / IMG_SIZE) * N_PATCHES
    row = (py / IMG_SIZE) * N_PATCHES
    return (np.clip(col, 0, N_PATCHES - 0.01), np.clip(row, 0, N_PATCHES - 0.01))

hand_df = pd.read_csv(HAND_CSV)
t0 = hand_df['tracking_timestamp_us'].iloc[0]
hand_df['time_sec'] = (hand_df['tracking_timestamp_us'] - t0) / 1e6
print(f"Loaded {len(hand_df)} hand tracking rows")

def nearest_hand_row(t_sec):
    idx = (hand_df['time_sec'] - t_sec).abs().idxmin()
    return hand_df.iloc[idx]


def gaussian_weight_map(points, sigma=SIGMA_PATCHES):
    gmap = np.zeros((N_PATCHES, N_PATCHES), dtype=np.float32)
    if not points:
        return gmap
    rows, cols = np.mgrid[0:N_PATCHES, 0:N_PATCHES].astype(np.float32)
    for (cx, cy) in points:
        gmap += np.exp(-((cols - cx)**2 + (rows - cy)**2) / (2 * sigma**2))
    if gmap.max() > 0:
        gmap = gmap / (gmap.max() + 1e-8)
    return gmap


def apply_hand_guidance(raw_attn, hand_row):
    patch_points = []
    pixel_points = []

    keypoints = [
        ('left_tracking_confidence', 'tx_left_wrist_device', 'ty_left_wrist_device', 'tz_left_wrist_device', 'left_wrist'),
        ('left_tracking_confidence', 'tx_left_palm_device', 'ty_left_palm_device', 'tz_left_palm_device', 'left_palm'),
        ('right_tracking_confidence', 'tx_right_wrist_device', 'ty_right_wrist_device', 'tz_right_wrist_device', 'right_wrist'),
        ('right_tracking_confidence', 'tx_right_palm_device', 'ty_right_palm_device', 'tz_right_palm_device', 'right_palm'),
    ]

    for conf_col, x_col, y_col, z_col, buf_name in keypoints:
        conf = float(hand_row.get(conf_col, 0))
        if conf < CONF_THRESHOLD:
            smooth_buf[buf_name].clear()
            continue

        tx = float(hand_row[x_col])
        ty = float(hand_row[y_col])
        tz = float(hand_row[z_col]) if z_col in hand_row.index else 0.5

        result = project_device_point_to_pixel(tx, ty, tz)
        if result is None:
            continue

        raw_px, raw_py = result

        smooth_buf[buf_name].append((raw_px, raw_py))
        sm_px = np.mean([p[0] for p in smooth_buf[buf_name]])
        sm_py = np.mean([p[1] for p in smooth_buf[buf_name]])

        if 0 <= sm_px < IMG_SIZE and 0 <= sm_py < IMG_SIZE:
            pixel_points.append((int(sm_px), int(sm_py)))
            patch_points.append(pixel_to_patch(sm_px, sm_py))

    hand_map = gaussian_weight_map(patch_points)
    any_hands = len(patch_points) > 0

    if any_hands:
        guided = raw_attn * (1.0 + GUIDE_STRENGTH * hand_map)
        guided = guided / (guided.sum() + 1e-8)
    else:
        guided = raw_attn.copy()

    return guided, hand_map, pixel_points, any_hands


def attn_entropy(a):
    p = a.flatten()
    p = p / (p.sum() + 1e-8)
    p = p[p > 0]
    return -np.sum(p * np.log(p + 1e-8))

def top_k_mass(a, k=10):
    f = a.flatten()
    return np.sort(f)[::-1][:k].sum() / (f.sum() + 1e-8)

def make_overlay(img_pil, attn_16x16):
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.
    attn_up = cv2.resize(attn_16x16, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    attn_n = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)
    heatmap = COLORMAP(attn_n)[:, :, :3]
    blended = (1 - BLEND_ALPHA) * img_np + BLEND_ALPHA * heatmap
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)

def draw_keypoints(img_np, pixel_points):
    out = img_np.copy()
    colors = [(0, 220, 80), (0, 180, 60), (30, 144, 255), (20, 100, 200)]
    for i, (px, py) in enumerate(pixel_points[:4]):
        cv2.circle(out, (px, py), 8, colors[i % len(colors)], -1)
        cv2.circle(out, (px, py), 8, (255, 255, 255), 1)
    return out


frame_files = sorted([f for f in os.listdir(FRAMES_FOLDER) if f.lower().endswith(('.jpg', '.png'))])
n_frames_total = len(frame_files)

selected_indices = list(range(0, n_frames_total, FRAME_STEP))
n_frames = len(selected_indices)

print(f"\nTotal frames: {n_frames_total}")
print(f"Processing every {FRAME_STEP}th frame → {n_frames} frames to process")
print(f"Output folder: {OUT_GUIDED}")

records_base = []
records_guided = []

for i, idx in enumerate(selected_indices):
    fname = frame_files[idx]
    t_sec = idx / VIDEO_FPS
    img = Image.open(os.path.join(FRAMES_FOLDER, fname)).convert('RGB')
    hand_row = nearest_hand_row(t_sec)

    raw_attn = get_cls_attention(img)
    guided_attn, hand_map, pixel_pts, any_hands = apply_hand_guidance(raw_attn, hand_row)

    m_base = {'frame_id': idx, 'timestamp': t_sec, 'entropy': attn_entropy(raw_attn),
              'top10_pct': top_k_mass(raw_attn) * 100}
    m_guided = {'frame_id': idx, 'timestamp': t_sec, 'entropy': attn_entropy(guided_attn),
                'top10_pct': top_k_mass(guided_attn) * 100, 'any_hands': int(any_hands),
                'n_keypoints': len(pixel_pts)}
    records_base.append(m_base)
    records_guided.append(m_guided)


    fig, axes = plt.subplots(1, 4, figsize=(20, 5), facecolor='#0d0d0d')
    for ax in axes:
        ax.set_facecolor('#0d0d0d')
        ax.axis('off')

    axes[0].imshow(img.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title(f'Original t={t_sec:.1f}s', color='white', fontsize=10)

    axes[1].imshow(make_overlay(img, raw_attn))
    axes[1].set_title(f'Baseline\nH={m_base["entropy"]:.2f}', color='white', fontsize=9)

    guided_overlay = make_overlay(img, guided_attn)
    if pixel_pts:
        guided_overlay = draw_keypoints(guided_overlay, pixel_pts)
    axes[2].imshow(guided_overlay)
    axes[2].set_title(f'Hand-guided\nH={m_guided["entropy"]:.2f}', color='white', fontsize=9)

    diff = guided_attn - raw_attn
    vmax = max(abs(diff.max()), abs(diff.min()), 1e-6)
    diff_up = cv2.resize(diff, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    axes[3].imshow(diff_up, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[3].set_title('Δ (guided - base)', color='white', fontsize=9)

    plt.tight_layout()
    fig.savefig(os.path.join(OUT_GUIDED, f'frame_{idx:04d}.png'), dpi=100, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{n_frames} frames (frame {idx})...")

print(f"\n✓ Saved {n_frames} visualizations → {OUT_GUIDED}/")


df_base = pd.DataFrame(records_base)
df_guided = pd.DataFrame(records_guided)
df_base.to_csv(f'{OUT_ROOT}/metrics_baseline.csv', index=False)
df_guided.to_csv(f'{OUT_ROOT}/metrics_guided.csv', index=False)

delta_H = df_guided['entropy'].mean() - df_base['entropy'].mean()
delta_top = df_guided['top10_pct'].mean() - df_base['top10_pct'].mean()

print(f"\n{'='*62}")
print("RESULTS SUMMARY — Hand-Guided Attention (same calibration as eye-gaze)")
print(f"Processed every {FRAME_STEP}th frame → {n_frames} frames")
print('='*62)
print(f"  Baseline avg entropy: {df_base['entropy'].mean():.4f}")
print(f"  Guided avg entropy: {df_guided['entropy'].mean():.4f}")
print(f"  Entropy change: {delta_H:+.4f}")
print(f"  Top-10 mass change: {delta_top:+.1f}%")
print(f"  Frames with hands detected: {df_guided['any_hands'].sum()}/{n_frames}")
print(f"  Avg keypoints per frame: {df_guided['n_keypoints'].mean():.2f}")
print('='*62)


print(f"\n{'='*62}")
print("PROJECTION DEBUG — First frame (using same calibration as eye-gaze)")
print('='*62)
first_idx = selected_indices[0] if selected_indices else 0
hr0 = nearest_hand_row(first_idx / VIDEO_FPS)
for label, conf_col, x_col, y_col, z_col in [
    ('Left wrist', 'left_tracking_confidence', 'tx_left_wrist_device', 'ty_left_wrist_device', 'tz_left_wrist_device'),
    ('Left palm', 'left_tracking_confidence', 'tx_left_palm_device', 'ty_left_palm_device', 'tz_left_palm_device'),
    ('Right wrist', 'right_tracking_confidence', 'tx_right_wrist_device', 'ty_right_wrist_device', 'tz_right_wrist_device'),
    ('Right palm', 'right_tracking_confidence', 'tx_right_palm_device', 'ty_right_palm_device', 'tz_right_palm_device'),
]:
    conf = float(hr0.get(conf_col, 0))
    tx = float(hr0.get(x_col, 0))
    ty = float(hr0.get(y_col, 0))
    tz = float(hr0.get(z_col, 0.5))
    proj = project_device_point_to_pixel(tx, ty, tz)
    in_frame = '✓ IN FRAME' if proj and 0 <= proj[0] < IMG_SIZE and 0 <= proj[1] < IMG_SIZE else '✗ OUTSIDE'
    if proj:
        print(f"  {label}: conf={conf:.2f}  ({tx:.3f},{ty:.3f},{tz:.3f}) → pixel=({proj[0]:.1f},{proj[1]:.1f}) {in_frame}")
    else:
        print(f"  {label}: conf={conf:.2f}  ({tx:.3f},{ty:.3f},{tz:.3f}) → projection FAILED")
print('='*62)

print(f"\n💡 TIP: To process ALL frames, change FRAME_STEP = 1 at the top")
print(f"   Current: FRAME_STEP = {FRAME_STEP}")


ZIP_PATH = '/content/hand_guided_final.zip'
print("\nCreating ZIP archive...")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT_GUIDED):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, BASE_DIR))
    for extra in [f'{OUT_ROOT}/metrics_baseline.csv', f'{OUT_ROOT}/metrics_guided.csv']:
        if os.path.exists(extra):
            zf.write(extra, os.path.relpath(extra, BASE_DIR))

files.download(ZIP_PATH)
print("✓ Download triggered")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Model loaded on GPU
Loading calibration from VRS (same as eye-gaze)...
  Camera: fx=1216.8, fy=1216.8, cx=1459.8, cy=1446.7
  Native size: 2880x2880
  Offset: X=4.39mm, Y=11.87mm
  Scaled: FXs=94.64, FYs=94.64, CXs=113.54, CYs=112.52
  Angular corrections: Yaw=5.00°, Pitch=2.74°
Loaded 329 hand tracking rows

Total frames: 330
Processing every 10th frame → 33 frames to process
Output folder: /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/pictures_corrected_final
  Processed 10/33 frames (frame 90)...
  Processed 20/33 frames (frame 190)...
  Processed 30/33 frames (frame 290)...

✓ Saved 33 visualizations → /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/pictures_corrected_final/

RESULTS SUMMARY — Hand-Guided Attention (same calibration as eye-gaze)
Processed every 10th frame → 33 frames
  Baseline avg entropy: 4.5992
  Guided avg entropy: 4.5920
  Entropy change: -0.0073
  Top-10 mass change: +0.2%
  Frames with hands detected: 5/33
  Avg 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered


In [ ]:


import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

BASE_DIR      = '/content/drive/MyDrive/Proiect_Dino/dino_wm'
FRAMES_FOLDER = f'{BASE_DIR}/datasets/raw_video/User15'
HAND_CSV      = f'{BASE_DIR}/datasets/wrist_and_palm_poses.csv'
JSONL_PATH    = f'{BASE_DIR}/datasets/online_calibration.jsonl'
OUT_ROOT      = f'{BASE_DIR}/datasets/attention_output'
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
NATIVE   = 2880

# ── Load calibration ──────────────────────────────────────────────────────────
with open(JSONL_PATH, 'r') as f:
    calib = json.loads(f.readline())
rgb_calib = next(c for c in calib['CameraCalibrations'] if c['Label'] == 'camera-rgb')
PARAMS = rgb_calib['Projection']['Params']
F_NAT, CX_NAT, CY_NAT = PARAMS[0], PARAMS[1], PARAMS[2]
K, P, S = PARAMS[3:9], PARAMS[9:11], PARAMS[11:15]
F  = F_NAT  * IMG_SIZE / NATIVE
CX = CX_NAT * IMG_SIZE / NATIVE
CY = CY_NAT * IMG_SIZE / NATIVE

tc   = rgb_calib['T_Device_Camera']
t_dc = np.array(tc['Translation'])
qw   = tc['UnitQuaternion'][0]
qx, qy, qz = tc['UnitQuaternion'][1]

def _quat_rot(qw, qx, qy, qz):
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw),  2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw),   1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw),   2*(qy*qz+qx*qw),   1-2*(qx**2+qy**2)],
    ])

R_dc = _quat_rot(qw, qx, qy, qz)
R_cd = R_dc.T
t_cd = -R_dc.T @ t_dc

def _project_raw(pt_device):
    """Project device-space 3D point → raw calibration pixel (u, v)."""
    Xc, Yc, Zc = R_cd @ pt_device + t_cd
    if Zc <= 0.02:
        return None
    r = np.sqrt(Xc**2 + Yc**2)
    theta = np.arctan2(r, Zc)
    t2 = theta**2
    td = theta * (1 + K[0]*t2 + K[1]*t2**2 + K[2]*t2**3 +
                      K[3]*t2**4 + K[4]*t2**5 + K[5]*t2**6)
    mx = (Xc/r)*td if r > 1e-9 else 0.
    my = (Yc/r)*td if r > 1e-9 else 0.
    r2 = mx**2 + my**2
    dx = 2*P[0]*mx*my + P[1]*(r2+2*mx**2)
    dy = P[0]*(r2+2*my**2) + 2*P[1]*mx*my
    sx = S[0]*r2 + S[1]*r2**2
    sy = S[2]*r2 + S[3]*r2**2
    return float(F*(mx+dx+sx)+CX), float(F*(my+dy+sy)+CY)

# ── Load hand data ────────────────────────────────────────────────────────────
df = pd.read_csv(HAND_CSV)
t0 = df['tracking_timestamp_us'].iloc[0]
df['time_sec'] = (df['tracking_timestamp_us'] - t0) / 1e6

frame_files = sorted([f for f in os.listdir(FRAMES_FOLDER)
                      if f.lower().endswith(('.jpg', '.png'))])
frame_idx = 10
t_sec     = frame_idx / 10.0
hand_row  = df.iloc[(df['time_sec'] - t_sec).abs().idxmin()]

img = np.array(Image.open(os.path.join(FRAMES_FOLDER, frame_files[frame_idx]))
               .convert('RGB').resize((IMG_SIZE, IMG_SIZE)))

raw_results = {}
for name, xc, yc, zc in [
    ('right_wrist', 'tx_right_wrist_device', 'ty_right_wrist_device', 'tz_right_wrist_device'),
    ('left_wrist',  'tx_left_wrist_device',  'ty_left_wrist_device',  'tz_left_wrist_device'),
]:
    pt = np.array([hand_row[xc], hand_row[yc], hand_row[zc]])
    raw_results[name] = _project_raw(pt)

variants = {
    'raw (no rotation)': lambda u, v: (u, v),
    'flip_X':            lambda u, v: (IMG_SIZE-u, v),
    'flip_Y':            lambda u, v: (u, IMG_SIZE-v),
    'flip_X_and_Y':      lambda u, v: (IMG_SIZE-u, IMG_SIZE-v),
    'rotate_90deg_CW':   lambda u, v: (IMG_SIZE-v, u),
    'rotate_90deg_CCW':  lambda u, v: (v, IMG_SIZE-u),
}

fig, axes = plt.subplots(1, 6, figsize=(24, 4), facecolor='#111')
fig.subplots_adjust(wspace=0.05)
colours = {'right_wrist': (0, 144, 255), 'left_wrist': (0, 220, 80)}
PRINCIPAL = (CX, CY)

for ax, (vname, tfn) in zip(axes, variants.items()):
    canvas = img.copy()
    ppx, ppy = tfn(PRINCIPAL[0], PRINCIPAL[1])
    ppx = int(np.clip(ppx, 0, IMG_SIZE-1))
    ppy = int(np.clip(ppy, 0, IMG_SIZE-1))
    cv2.drawMarker(canvas, (ppx, ppy), (255, 80, 80), cv2.MARKER_CROSS, 14, 2)
    for kname, raw in raw_results.items():
        if raw is None:
            continue
        ux, vy = tfn(raw[0], raw[1])
        ux, vy = int(round(ux)), int(round(vy))
        col = colours[kname]
        in_f = 0 <= ux < IMG_SIZE and 0 <= vy < IMG_SIZE
        ux2 = int(np.clip(ux, 0, IMG_SIZE-1))
        vy2 = int(np.clip(vy, 0, IMG_SIZE-1))
        cv2.circle(canvas, (ux2, vy2), 9, col, -1)
        cv2.circle(canvas, (ux2, vy2), 9, (255, 255, 255), 1)
        if not in_f:
            cv2.putText(canvas, 'OUT', (ux2, vy2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 0), 1)
    ax.imshow(canvas)
    ax.set_title(vname, color='white', fontsize=8, pad=4)
    ax.axis('off')

fig.suptitle(
    f'ROTATION DIAGNOSTIC — frame {frame_idx} (t={t_sec:.1f}s)\n'
    f'RED cross = principal point  |  GREEN = left wrist  |  BLUE = right wrist\n'
    f'Pick the panel where dots land ON the actual hands.',
    color='white', fontsize=9, y=1.08,
)
diag_path = f'{OUT_ROOT}/rotation_diagnostic.png'
fig.savefig(diag_path, dpi=130, bbox_inches='tight', facecolor='#111')
plt.close(fig)
print(f"✓ Saved diagnostic → {diag_path}")



FRAME_ROTATION = 'rot_cw'   # 'none' | 'flip_x' | 'flip_y' | 'flip_xy' | 'rot_cw' | 'rot_ccw'
VIDEO_FPS      = 10

SIGMA_PATCHES  = 2.5
GUIDE_STRENGTH = 15.0
CONF_THRESHOLD = 0.5
FRAME_STEP     = 10

BLEND_ALPHA    = 0.55
COLORMAP       = plt.cm.inferno
N_PATCHES      = 16

def apply_rotation_correction(u_cal, v_cal, mode=FRAME_ROTATION):
    if   mode == 'none':    return u_cal,            v_cal
    elif mode == 'flip_x':  return IMG_SIZE - u_cal, v_cal
    elif mode == 'flip_y':  return u_cal,            IMG_SIZE - v_cal
    elif mode == 'flip_xy': return IMG_SIZE - u_cal, IMG_SIZE - v_cal
    elif mode == 'rot_cw':  return IMG_SIZE - v_cal, u_cal
    elif mode == 'rot_ccw': return v_cal,            IMG_SIZE - u_cal
    else: raise ValueError(f"Unknown FRAME_ROTATION: {mode!r}")

print(f"Config: VIDEO_FPS={VIDEO_FPS}, FRAME_ROTATION='{FRAME_ROTATION}'")
print(f"        GUIDE_STRENGTH={GUIDE_STRENGTH}, SIGMA_PATCHES={SIGMA_PATCHES}")



import torch
from transformers import AutoImageProcessor, AutoModel, AutoConfig
from collections import deque

print("\nLoading DINOv2 …")
_model_name = "facebook/dinov2-base"
_proc  = AutoImageProcessor.from_pretrained(_model_name)
_cfg   = AutoConfig.from_pretrained(_model_name)
_cfg.output_attentions = True
_model = AutoModel.from_pretrained(_model_name, config=_cfg).eval()
if torch.cuda.is_available():
    _model = _model.cuda()
    print("  GPU")
else:
    print("  CPU")

def get_cls_attention(img_pil):
    inputs = _proc(images=img_pil, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        out = _model(**inputs)
    if not hasattr(out, 'attentions') or out.attentions is None:
        m = np.ones((N_PATCHES, N_PATCHES), dtype=np.float32)
        return m / m.sum()
    attn = out.attentions[-1][0, :, 0, 1:257].mean(0)
    try:
        m = attn.reshape(N_PATCHES, N_PATCHES).cpu().numpy()
    except Exception:
        s = int(np.sqrt(attn.shape[0]))
        m = attn[:s*s].reshape(s, s).cpu().numpy()
        m = cv2.resize(m, (N_PATCHES, N_PATCHES), interpolation=cv2.INTER_CUBIC)
    return m / (m.sum() + 1e-8)

print("  Model ready.")




# Smoothing buffers (3-frame rolling average per keypoint)
_smooth = {k: deque(maxlen=3) for k in
           ['left_wrist', 'left_palm', 'right_wrist', 'right_palm']}

# (confidence_column, x_col, y_col, z_col, buffer_key, draw_colour)
_KEYPOINTS = [
    ('left_tracking_confidence',
     'tx_left_wrist_device',  'ty_left_wrist_device',  'tz_left_wrist_device',
     'left_wrist',  (0, 220, 80)),
    ('left_tracking_confidence',
     'tx_left_palm_device',   'ty_left_palm_device',   'tz_left_palm_device',
     'left_palm',   (0, 180, 60)),
    ('right_tracking_confidence',
     'tx_right_wrist_device', 'ty_right_wrist_device', 'tz_right_wrist_device',
     'right_wrist', (30, 144, 255)),
    ('right_tracking_confidence',
     'tx_right_palm_device',  'ty_right_palm_device',  'tz_right_palm_device',
     'right_palm',  (20, 100, 200)),
]

def _project_to_stored_pixel(pt_device):
    """device-space 3D → fisheye projection → rotation correction → pixel."""
    raw = _project_raw(pt_device)
    if raw is None:
        return None
    u, v = apply_rotation_correction(raw[0], raw[1])
    return float(u), float(v)

def _gaussian_map(patch_points):
    gmap = np.zeros((N_PATCHES, N_PATCHES), dtype=np.float32)
    if not patch_points:
        return gmap
    rows, cols = np.mgrid[0:N_PATCHES, 0:N_PATCHES].astype(np.float32)
    for (pc, pr) in patch_points:
        gmap += np.exp(-((cols - pc)**2 + (rows - pr)**2) / (2 * SIGMA_PATCHES**2))
    return gmap / (gmap.max() + 1e-8)

def apply_hand_guidance(raw_attn, hand_row):
    """
    Returns (guided_attn, hand_map, pixel_pts, any_hands).
    Uses _smooth, _KEYPOINTS, _project_to_stored_pixel — all defined above.
    """
    patch_pts, pixel_pts = [], []
    for conf_col, xc, yc, zc, buf_key, _ in _KEYPOINTS:
        conf = float(hand_row.get(conf_col, 0))
        if conf < CONF_THRESHOLD:
            _smooth[buf_key].clear()
            continue
        pt  = np.array([float(hand_row[xc]), float(hand_row[yc]), float(hand_row[zc])])
        res = _project_to_stored_pixel(pt)
        if res is None:
            continue
        u, v = res
        _smooth[buf_key].append((u, v))
        u = float(np.mean([p[0] for p in _smooth[buf_key]]))
        v = float(np.mean([p[1] for p in _smooth[buf_key]]))
        if not (0 <= u < IMG_SIZE and 0 <= v < IMG_SIZE):
            continue
        pixel_pts.append((int(u), int(v)))
        pc = float(np.clip((u / IMG_SIZE) * N_PATCHES, 0, N_PATCHES - 0.01))
        pr = float(np.clip((v / IMG_SIZE) * N_PATCHES, 0, N_PATCHES - 0.01))
        patch_pts.append((pc, pr))

    hand_map  = _gaussian_map(patch_pts)
    any_hands = len(patch_pts) > 0
    if any_hands:
        guided = raw_attn * (1.0 + GUIDE_STRENGTH * hand_map)
        guided = guided / (guided.sum() + 1e-8)
    else:
        guided = raw_attn.copy()
    return guided, hand_map, pixel_pts, any_hands

print("  Cell 3 ready: _smooth, _KEYPOINTS, apply_hand_guidance all defined ✓")


import zipfile
from google.colab import files

OUT_GUIDED = f'{OUT_ROOT}/pictures_corrected'
Path(OUT_GUIDED).mkdir(parents=True, exist_ok=True)

def _overlay(img_pil, attn):
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.
    up = cv2.resize(attn, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    n  = (up - up.min()) / (up.max() - up.min() + 1e-8)
    blended = (1 - BLEND_ALPHA) * img_np + BLEND_ALPHA * COLORMAP(n)[:, :, :3]
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)

def _draw_kpts(arr, px_pts):
    out  = arr.copy()
    cols = [(0, 220, 80), (0, 180, 60), (30, 144, 255), (20, 100, 200)]
    for i, (px, py) in enumerate(px_pts[:4]):
        cv2.circle(out, (px, py), 9, cols[i % len(cols)], -1)
        cv2.circle(out, (px, py), 9, (255, 255, 255), 1)
    return out

def _entropy(a):
    p = a.flatten(); p = p / (p.sum() + 1e-8); p = p[p > 0]
    return float(-np.sum(p * np.log(p + 1e-8)))

def _top10(a):
    f = a.flatten()
    return float(np.sort(f)[::-1][:10].sum() / (f.sum() + 1e-8))

frame_files = sorted([f for f in os.listdir(FRAMES_FOLDER)
                      if f.lower().endswith(('.jpg', '.png'))])
N_total = len(frame_files)
indices = list(range(0, N_total, FRAME_STEP))

print(f"\nFrames: {N_total} total, every {FRAME_STEP}th → {len(indices)} frames")
print(f"VIDEO_FPS={VIDEO_FPS}, FRAME_ROTATION='{FRAME_ROTATION}'\n")

records_base, records_guided = [], []

for i, idx in enumerate(indices):
    fname    = frame_files[idx]
    t_sec    = idx / VIDEO_FPS
    img      = Image.open(os.path.join(FRAMES_FOLDER, fname)).convert('RGB')
    hand_row = df.iloc[(df['time_sec'] - t_sec).abs().idxmin()]

    raw_attn = get_cls_attention(img)
    guided, hand_map, px_pts, hands = apply_hand_guidance(raw_attn, hand_row)

    m_b = dict(frame_id=idx, timestamp=t_sec,
               entropy=_entropy(raw_attn), top10_pct=_top10(raw_attn)*100)
    m_g = dict(frame_id=idx, timestamp=t_sec,
               entropy=_entropy(guided), top10_pct=_top10(guided)*100,
               any_hands=int(hands), n_keypoints=len(px_pts))
    records_base.append(m_b)
    records_guided.append(m_g)

    # ── 4-panel figure ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), facecolor='#0d0d0d')
    for ax in axes:
        ax.set_facecolor('#0d0d0d'); ax.axis('off')

    axes[0].imshow(img.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title(f'Original  t={t_sec:.1f}s', color='white', fontsize=10, pad=5)

    axes[1].imshow(_overlay(img, raw_attn))
    axes[1].set_title(
        f'Baseline DINOv2\nH={m_b["entropy"]:.3f}  top10={m_b["top10_pct"]:.0f}%',
        color='#aaaaaa', fontsize=9, pad=5)

    ov     = _overlay(img, guided)
    if px_pts:
        ov = _draw_kpts(ov, px_pts)
    label  = f'HANDS ({len(px_pts)} pts)' if hands else 'no hands in frame'
    colour = '#7fffb2' if hands else '#ff7043'
    axes[2].imshow(ov)
    axes[2].set_title(
        f'Hand-guided  [{label}]\nH={m_g["entropy"]:.3f}  top10={m_g["top10_pct"]:.0f}%',
        color=colour, fontsize=9, pad=5)

    diff    = guided - raw_attn
    vmax    = max(abs(diff.max()), abs(diff.min()), 1e-6)
    diff_up = cv2.resize(diff, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    dH      = m_g['entropy'] - m_b['entropy']
    axes[3].imshow(diff_up, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[3].set_title(f'Δ (guided − baseline)\nΔH={dH:+.3f}', color='white', fontsize=9, pad=5)

    fig.suptitle(
        f'Frame {idx:04d} — {fname}  |  '
        f'GUIDE_STRENGTH={GUIDE_STRENGTH}  σ={SIGMA_PATCHES}  '
        f'fps={VIDEO_FPS}  rot={FRAME_ROTATION}',
        color='#cccccc', fontsize=9, y=1.01)
    plt.tight_layout(pad=0.3)
    fig.savefig(os.path.join(OUT_GUIDED, f'frame_{idx:04d}.png'),
                dpi=100, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)

    if (i+1) % 5 == 0 or (i+1) == len(indices):
        print(f"  [{i+1:3d}/{len(indices)}]  frame={idx:04d}  t={t_sec:.1f}s  "
              f"H_base={m_b['entropy']:.3f}  H_guided={m_g['entropy']:.3f}  "
              f"ΔH={dH:+.3f}  hands={'✓' if hands else '✗'}")

print(f"\n✓ Saved {len(indices)} frames → {OUT_GUIDED}/")

# ── Save CSVs ─────────────────────────────────────────────────────────────────
df_b = pd.DataFrame(records_base)
df_g = pd.DataFrame(records_guided)
df_b.to_csv(f'{OUT_ROOT}/metrics_baseline.csv', index=False)
df_g.to_csv(f'{OUT_ROOT}/metrics_guided.csv',   index=False)

delta_H   = df_g['entropy'].mean()   - df_b['entropy'].mean()
delta_top = df_g['top10_pct'].mean() - df_b['top10_pct'].mean()

print(f"\n{'='*62}")
print(f"RESULTS  (fps={VIDEO_FPS}  rotation={FRAME_ROTATION})")
print('='*62)
print(f"  {'Metric':<38} {'Baseline':>9} {'Guided':>9} {'Δ':>7}")
print(f"  {'-'*60}")
for label, b_val, g_val in [
    ('Mean entropy (lower=focused)',    df_b['entropy'].mean(),  df_g['entropy'].mean()),
    ('Std  entropy',                    df_b['entropy'].std(),   df_g['entropy'].std()),
    ('Min  entropy (best frame)',       df_b['entropy'].min(),   df_g['entropy'].min()),
]:
    print(f"  {label:<38} {b_val:>9.4f} {g_val:>9.4f} {g_val-b_val:>+7.4f}")
print(f"  {'Mean top-10 mass % (higher=sharper)':<38} "
      f"{df_b['top10_pct'].mean():>8.1f}% {df_g['top10_pct'].mean():>8.1f}% {delta_top:>+6.1f}%")
print(f"  {'Frames with hands in frame':<38} "
      f"{'—':>9} {df_g['any_hands'].mean()*100:>8.1f}%")
print('='*62)

# ── ZIP and download ──────────────────────────────────────────────────────────
ZIP_PATH = '/content/hand_guided_fixed.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT_GUIDED):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, BASE_DIR))
    for extra in [f'{OUT_ROOT}/rotation_diagnostic.png',
                  f'{OUT_ROOT}/metrics_baseline.csv',
                  f'{OUT_ROOT}/metrics_guided.csv']:
        if os.path.exists(extra):
            zf.write(extra, os.path.relpath(extra, BASE_DIR))
files.download(ZIP_PATH)
print(f"✓ Download triggered: {ZIP_PATH}")

✓ Saved diagnostic → /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/rotation_diagnostic.png
Config: VIDEO_FPS=10, FRAME_ROTATION='rot_cw'
        GUIDE_STRENGTH=15.0, SIGMA_PATCHES=2.5

Loading DINOv2 …


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  GPU
  Model ready.
  Cell 3 ready: _smooth, _KEYPOINTS, apply_hand_guidance all defined ✓

Frames: 330 total, every 10th → 33 frames
VIDEO_FPS=10, FRAME_ROTATION='rot_cw'

  [  5/33]  frame=0040  t=4.0s  H_base=4.707  H_guided=4.392  ΔH=-0.315  hands=✓
  [ 10/33]  frame=0090  t=9.0s  H_base=4.276  H_guided=3.115  ΔH=-1.161  hands=✓
  [ 15/33]  frame=0140  t=14.0s  H_base=4.634  H_guided=4.153  ΔH=-0.481  hands=✓
  [ 20/33]  frame=0190  t=19.0s  H_base=4.443  H_guided=3.638  ΔH=-0.805  hands=✓
  [ 25/33]  frame=0240  t=24.0s  H_base=4.628  H_guided=4.131  ΔH=-0.497  hands=✓
  [ 30/33]  frame=0290  t=29.0s  H_base=4.655  H_guided=4.080  ΔH=-0.574  hands=✓
  [ 33/33]  frame=0320  t=32.0s  H_base=4.796  H_guided=4.417  ΔH=-0.380  hands=✓

✓ Saved 33 frames → /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/pictures_corrected/

RESULTS  (fps=10  rotation=rot_cw)
  Metric                                  Baseline    Guided       Δ
  ------------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered: /content/hand_guided_fixed.zip


In [ ]:
!pip install projectaria-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.4 MB/s eta 0:00:00
  Attempting uninstall: moviepy
    Found existing installation: moviepy 1.0.3
    Uninstalling moviepy-1.0.3:
      Successfully uninstalled moviepy-1.0.3


In [ ]:
!rm -rf /content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/attention_output/pictures_corrected*

In [ ]:
# ── DIAGNOSTIC CELL — run this and paste the output ──────────────────────────
import pandas as pd
import numpy as np

HAND_CSV = '/content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/wrist_and_palm_poses.csv'
hand_df  = pd.read_csv(HAND_CSV)

print("=== COLUMNS ===")
print(list(hand_df.columns))

print("\n=== FIRST 3 ROWS (transposed) ===")
print(hand_df.head(3).T.to_string())

print("\n=== VALUE RANGES (non-timestamp cols) ===")
for col in hand_df.columns:
    if 'timestamp' in col: continue
    print(f"  {col:45s}  min={hand_df[col].min():>10.4f}  max={hand_df[col].max():>10.4f}  "
          f"mean={hand_df[col].mean():>10.4f}")

# Also inspect the SE3 transform methods
print("\n=== SE3 AVAILABLE METHODS ===")
from projectaria_tools.core import data_provider
VRS_PATH = '/content/drive/MyDrive/Proiect_Dino/dino_wm/datasets/User_15_Short_10.vrs'
provider = data_provider.create_vrs_data_provider(VRS_PATH)
device_calib = provider.get_device_calibration()
T = device_calib.get_transform_device_sensor("camera-rgb")
methods = [m for m in dir(T) if not m.startswith('_')]
print(methods)

# Try the matrix
print("\n=== SE3 MATRIX (full 4x4) ===")
try:
    print(T.to_matrix())
except: pass
try:
    print(T.matrix())
except: pass

print("\n=== ROTATION ===")
try:
    print("rotation():", T.rotation())
except: pass
try:
    r = T.rotation()
    print("rotation methods:", [m for m in dir(r) if not m.startswith('_')])
    print("to_matrix():", r.to_matrix())
except Exception as e:
    print("Error:", e)

=== COLUMNS ===
['tracking_timestamp_us', 'left_tracking_confidence', 'tx_left_wrist_device', 'ty_left_wrist_device', 'tz_left_wrist_device', 'tx_left_palm_device', 'ty_left_palm_device', 'tz_left_palm_device', 'right_tracking_confidence', 'tx_right_wrist_device', 'ty_right_wrist_device', 'tz_right_wrist_device', 'tx_right_palm_device', 'ty_right_palm_device', 'tz_right_palm_device', 'nx_left_palm_device', 'ny_left_palm_device', 'nz_left_palm_device', 'nx_left_wrist_device', 'ny_left_wrist_device', 'nz_left_wrist_device', 'nx_right_palm_device', 'ny_right_palm_device', 'nz_right_palm_device', 'nx_right_wrist_device', 'ny_right_wrist_device', 'nz_right_wrist_device']

=== FIRST 3 ROWS (transposed) ===
                                      0             1             2
tracking_timestamp_us      9.228755e+11  9.228756e+11  9.228757e+11
left_tracking_confidence   9.995030e-01  9.998080e-01  9.998290e-01
tx_left_wrist_device       4.652860e-01  4.679290e-01  4.632760e-01
ty_left_wrist_devi

In [ ]:
import os

# Vezi cum se numeste notebook-ul curent in Colab
os.system("ls /content/drive/MyDrive/Colab\\ Notebooks/")

# Sau cauta in tot Drive-ul
os.system("find /content/drive/MyDrive -name '*.ipynb' 2>/dev/null")

256